# Quantum Subset Sum Solver

This project implements a quantum algorithm to solve the Subset Sum problem using a combination of the Draper Quantum Fourier Transform (QFT) adder and Grover's search algorithm. It is built entirely in Qiskit.

## Algorithm Overview

The solver uses a quantum oracle to evaluate all possible subsets simultaneously using superposition. It adds the values of the included subset elements into an accumulator register in the Fourier basis. If the accumulator matches the desired target value $t$, a phase kickback flips the sign of the corresponding basis state. Finally, a Grover diffusion operator amplifies the probability amplitude of the marked state(s).

The circuit architecture consists of $n$ index qubits (representing subset inclusion), $m$ accumulator qubits where $m = \lceil \log_2(\sum x_i) \rceil$, and $1$ flag qubit for phase kickback.

## Mathematical Formalism

### 1. State Initialization
We initialize the $n$ index qubits in a uniform superposition:
$$
| \psi_0 \rangle = H^{\otimes n} |0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} |k\rangle
$$
Each computational basis state $|k\rangle$ corresponds to a unique subset of the input array.

### 2. The Oracle (Draper QFT Adder)
To compute the sum without destroying the superposition, we utilize a Draper adder on the accumulator register (initially $|0\rangle$).

We apply the QFT to the accumulator:
$$
\text{QFT} |0\rangle^{\otimes m} = \frac{1}{\sqrt{2^m}} \sum_{y=0}^{2^m-1} |y\rangle
$$

Then, for each element $x_i$ in the input array, controlled by the $i$-th index qubit, we apply controlled-phase shifts to the accumulator. The phase applied to the $j$-th qubit (where $j=0$ is the most significant phase bit in Qiskit's convention, or $j \in [0, m-1]$ from LSB to MSB in standard binary notation) is defined by the rotation operator:
$$
R_Z \left( \frac{2\pi x_i}{2^{m-j}} \right)
$$

These rotations effectively add $x_i$ to the accumulator in the phase representation:
$$
|y\rangle \rightarrow \exp\left( \frac{2\pi i \cdot x_i \cdot y}{2^m} \right) |y\rangle
$$

An Inverse QFT (IQFT) maps the accumulator back to the computational basis, revealing the exact sum $s_k$ for each subset $|k\rangle$.

### 3. Phase Kickback and Uncomputation
We verify if the computed sum $s_k$ equals the target $t$. We apply $X$ gates framing the bitwise limits of $t$:
$$
X^{\text{where } t_j = 0}
$$
A multi-controlled $X$ (MCX) gate targets the flag qubit (initialized to $|-\rangle$), applying a $-1$ phase shift if the sum equals $t$.

We then precisely reverse the Oracle operations (QFT $\rightarrow$ Inverse Phase Rotations $\rightarrow$ IQFT) to perfectly uncompute the accumulator, returning it to $|0\rangle^{\otimes m}$ while preserving quantum interference and avoiding entanglement between the index and accumulator registers.

### 4. Grover Diffusion Operator
The standard amplitude amplification operator is applied to the index register:
$$
U_s = 2|s\rangle\langle s| - I
$$
Where $|s\rangle$ is the uniform superposition state.

### Complexity
The quantum algorithm finds a solution in $\mathcal{O}(\sqrt{2^n})$ evaluations, representing a quadratic speedup over classical brute-force search ($\mathcal{O}(2^n)$).

## Benchmarking & Validation

The algorithm incorporates a headless testing suite constructed via `pytest`. The validation framework uses the `Qiskit Aer` multi-threaded simulator to handle dense statevector calculations.

The suite verifies:
- **Trivial cases:** Small inputs ($n=2$) to ensure baseline phase alignment.
- **Boundary cases:** Maximum bounds (e.g. $n=4, x_i=7, t=28$).
- **Null cases:** Unsolvable targets result in diffuse, near-uniform probability distributions instead of collapsing to a false positive.

### Running Tests
To run the automated tests locally, use `pytest`:
```bash
pytest test_subset_sum.py -v
```
To achieve optimal statevector benchmarking performance, the backend strictly configures CPU threading.

## Interactive Demonstration

In addition to the headless testing suite, the project includes an interactive demonstration script.

You can run the script using Python:
```bash
python run_demo.py
```

The `run_demo.py` script allows you to:
1. Interactively input a custom array of integers and a target sum.
2. Build and execute the full Grover-QFT subset sum circuit on the Qiskit Aer backend.
3. Automatically parse the results and translate the highest probability state back into human-readable subset values.
4. Generate and save a visual bar chart (`histogram_results.png`) plotting the subset probability distribution (requires `matplotlib`).

## Core Quantum Algorithm
The following cell implements the Draper QFT Adder, Grover diffusers, and circuit builder.

In [ ]:
import math
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT

def get_aer_simulator(threads: int = 4):
    """
    Returns a Qiskit Aer simulator explicitly configured for multithreading.
    """
    backend = AerSimulator(
        max_parallel_threads=threads,
        max_parallel_experiments=1,
        max_parallel_shots=1,
        statevector_parallel_threshold=10
    )
    return backend

def initialize_state(n: int, m: int):
    """
    Allocates n index qubits, m accumulator qubits, and 1 flag qubit.
    Applies Hadamard gates to the index register.
    """
    index_qubits = QuantumRegister(n, 'idx')
    accumulator_qubits = QuantumRegister(m, 'acc')
    flag_qubit = QuantumRegister(1, 'flag')
    
    qc = QuantumCircuit(index_qubits, accumulator_qubits, flag_qubit)
    
    # Initialize index qubits in uniform superposition
    qc.h(index_qubits)
    
    # Initialize flag qubit in |-> state
    qc.x(flag_qubit)
    qc.h(flag_qubit)
    
    return qc, index_qubits, accumulator_qubits, flag_qubit

def draper_qft_adder(qc: QuantumCircuit, index_qubits: QuantumRegister, accumulator_qubits: QuantumRegister, numbers: list[int], inverse: bool = False):
    """
    Applies the Oracle Core logic: Draper QFT adder.
    Adds the subset sum into the accumulator in the Fourier basis.
    """
    m = len(accumulator_qubits)
    n = len(index_qubits)
    
    if not inverse:
        # Apply QFT to accumulator
        qc.append(QFT(m, do_swaps=True).to_instruction(), accumulator_qubits)
        
        # Controlled-phase rotations
        for i in range(n):
            val = numbers[i]
            for j in range(m):
                phase = 2 * math.pi * val / (2 ** (m - j))
                if phase % (2*math.pi) != 0:
                    qc.cp(phase, index_qubits[i], accumulator_qubits[j])
        
        # Apply Inverse QFT
        qc.append(QFT(m, do_swaps=True, inverse=True).to_instruction(), accumulator_qubits)
    else:
        # Inverse operation: QFT -> Inv Phase Rotations -> IQFT
        qc.append(QFT(m, do_swaps=True).to_instruction(), accumulator_qubits)
        
        for i in reversed(range(n)):
            val = numbers[i]
            for j in reversed(range(m)):
                phase = -2 * math.pi * val / (2 ** (m - j))
                if phase % (2*math.pi) != 0:
                    qc.cp(phase, index_qubits[i], accumulator_qubits[j])
        
        qc.append(QFT(m, do_swaps=True, inverse=True).to_instruction(), accumulator_qubits)

def phase_kickback_and_uncompute(qc: QuantumCircuit, index_qubits: QuantumRegister, accumulator_qubits: QuantumRegister, flag_qubit: QuantumRegister, numbers: list[int], target: int):
    """
    Applies X gates framing the binary representation of target state t.
    Executes MCX gate across accumulator targeting the flag qubit.
    Uncomputes the accumulator back to |0>.
    """
    m = len(accumulator_qubits)
    
    # 1. Flip accumulator qubits where the target binary representation is 0.
    target_bin = bin(target)[2:].zfill(m)
    for j in range(m):
        bit_val = target_bin[m - 1 - j]
        if bit_val == '0':
            qc.x(accumulator_qubits[j])
            
    # 2. MCX gate across accumulator targeting flag
    qc.mcx(accumulator_qubits, flag_qubit[0])
    
    # 3. Un-flip the accumulator qubits
    for j in range(m):
        bit_val = target_bin[m - 1 - j]
        if bit_val == '0':
            qc.x(accumulator_qubits[j])
            
    # 4. Uncompute the addition
    draper_qft_adder(qc, index_qubits, accumulator_qubits, numbers, inverse=True)

def grover_diffuser(qc: QuantumCircuit, index_qubits: QuantumRegister):
    """
    Amplitude amplification operator 2|s><s| - I on index register.
    """
    n = len(index_qubits)
    qc.h(index_qubits)
    qc.x(index_qubits)
    
    qc.h(index_qubits[n-1])
    if n > 1:
        qc.mcx(index_qubits[:-1], index_qubits[n-1])
    else:
        qc.x(index_qubits[0])
    qc.h(index_qubits[n-1])
    
    qc.x(index_qubits)
    qc.h(index_qubits)

def build_subset_sum_circuit(numbers: list[int], target: int) -> QuantumCircuit:
    n = len(numbers)
    max_sum = sum(numbers)
    if max_sum == 0:
        m = 1
    else:
        m = math.ceil(math.log2(max_sum + 1))
        
    if target > max_sum or target < 0:
        m = max(m, math.ceil(math.log2(abs(target) + 1)))

    qc, index_qubits, accumulator_qubits, flag_qubit = initialize_state(n, m)
    
    # Optimal number of Grover iterations roughly (pi/4) * sqrt(N/M)
    # Here N = 2^n. M is unknown, but we assume 1.
    iterations = max(1, round(math.pi / 4 * math.sqrt(2**n)))
    
    for _ in range(iterations):
        draper_qft_adder(qc, index_qubits, accumulator_qubits, numbers, inverse=False)
        phase_kickback_and_uncompute(qc, index_qubits, accumulator_qubits, flag_qubit, numbers, target)
        grover_diffuser(qc, index_qubits)
        
    cr = ClassicalRegister(n, 'c')
    qc.add_register(cr)
    
    # Only measure the index qubits. Keep the bit order aligned.
    qc.measure(index_qubits, cr)
    
    return qc

## Interactive Demonstration
Run the cell below to test the circuit. It will prompt you for an array of integers and a target sum.

In [ ]:
import os
from qiskit import transpile
from qiskit.visualization import plot_histogram

def run_demonstration():
    print("--- Quantum Subset Sum Demonstration ---")
    print("Please enter a comma-separated list of positive integers for the array:")
    arr_input = input("> ").strip()
    if not arr_input:
        print("No input provided. Using default array: 2, 3, 5, 7")
        numbers = [2, 3, 5, 7]
    else:
        numbers = [int(x.strip()) for x in arr_input.split(",")]
        
    print("Please enter the target sum (integer):")
    target_input = input("> ").strip()
    if not target_input:
        print("No target provided. Using default target: 10")
        target = 10
    else:
        target = int(target_input)
        
    shots = 1024
    
    print(f"Input Array: {numbers}")
    print(f"Target Sum:  {target}")
    print("\nBuilding quantum circuit...")
    
    qc = build_subset_sum_circuit(numbers, target)
    backend = get_aer_simulator(threads=4)
    
    print(f"Executing on Qiskit Aer Simulator ({shots} shots)...")
    t_qc = transpile(qc, backend)
    
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    counts = result.get_counts()
    
    # Analyze the results
    most_probable_state = max(counts, key=counts.get)
    highest_prob = counts[most_probable_state] / shots
    
    print("\n--- Execution Results ---")
    if highest_prob > 0.5:
        # Translate the measured bitstring back into the subset
        # Qiskit orders bitstrings from q_{n-1} (left) to q_0 (right).
        subset = []
        for i, bit in enumerate(reversed(most_probable_state)):
            if bit == '1':
                subset.append(numbers[i])
                
        print(f"Success! Found valid subset: {subset}")
        print(f"Measured state '{most_probable_state}' with {highest_prob*100:.1f}% probability.")
        print(f"Subset sum validates to: {sum(subset)}")
    else:
        print("No valid subset found with high probability.")
        print(f"The distribution is diffuse. Highest probability state '{most_probable_state}' was only {highest_prob*100:.1f}%.")
        
    print("\nGenerating histogram plot...")
    try:
        fig = plot_histogram(counts, title=f"Subset Sum Probability Distribution\nTarget: {target} | Array: {numbers}")
        filename = "histogram_results.png"
        fig.savefig(filename)
        print(f"Histogram successfully saved as '{filename}' in the current directory.")
    except ImportError:
        print("Note: 'matplotlib' is required to save the histogram image. Please install it using 'pip install matplotlib'.")

if __name__ == "__main__":
    run_demonstration()

## Testing Suite
Verify edge cases and expected failure states.

In [ ]:
import pytest
from qiskit import transpile

def run_subset_sum_experiment(numbers, target, shots=1000):
    qc = build_subset_sum_circuit(numbers, target)
    backend = get_aer_simulator(threads=4)
    
    t_qc = transpile(qc, backend)
    
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    counts = result.get_counts()
    
    return counts

def test_trivial_case():
    """
    n=2, numbers=[1, 2], target=3. 
    Valid subset: [1, 2] -> index '11'
    """
    numbers = [1, 2]
    target = 3
    counts = run_subset_sum_experiment(numbers, target, shots=1000)
    
    assert '11' in counts
    prob = counts.get('11', 0) / 1000.0
    assert prob > 0.9

def test_boundary_case():
    """
    n=4, numbers=[7, 7, 7, 7], target=28.
    Valid subset: [7, 7, 7, 7] -> index '1111'
    """
    numbers = [7, 7, 7, 7]
    target = 28
    counts = run_subset_sum_experiment(numbers, target, shots=1000)
    
    assert '1111' in counts
    prob = counts.get('1111', 0) / 1000.0
    assert prob > 0.9

def test_null_case():
    """
    n=3, numbers=[2, 4, 6], target=5
    No subset sums to 5.
    """
    numbers = [2, 4, 6]
    target = 5
    counts = run_subset_sum_experiment(numbers, target, shots=1000)
    
    max_count = max(counts.values())
    max_prob = max_count / 1000.0
    assert max_prob < 0.5 


print('Running tests...')
test_trivial_case()
test_boundary_case()
test_null_case()
print('All tests passed!')